# Lesson 2.1 - AKS Cluster And GPU Node Pool

**Level: Intermediate**

## Learning Objectives
- Build an AKS cluster with Azure CNI overlay.
- Add an optional GPU node pool for vLLM.
- Understand taints, tolerations, and workload placement.

**Project file(s) produced:** `terraform/foundation/aks.tf`

## Concepts
The AKS resource defines control plane settings and the system node pool.
A second node pool is created only when `gpu_node_pool_enabled = true`.
GPU nodes are tainted so only explicitly tolerant pods run there.

In [ ]:
%%writefile terraform/foundation/aks.tf
resource "azurerm_kubernetes_cluster" "this" {
  name                = local.names.aks
  resource_group_name = local.resource_group_name
  location            = local.rg_location
  dns_prefix          = local.names.aks
  kubernetes_version  = var.kubernetes_version

  default_node_pool {
    name                 = "system"
    vm_size              = var.system_node_pool.vm_size
    vnet_subnet_id       = module.vnet.subnets["aks"].resource_id
    auto_scaling_enabled = true
    node_count           = var.system_node_pool.node_count
    min_count            = var.system_node_pool.min_count
    max_count            = var.system_node_pool.max_count
  }

  identity {
    type = "SystemAssigned"
  }

  network_profile {
    network_plugin      = "azure"
    network_plugin_mode = "overlay"
    pod_cidr            = "10.244.0.0/16"
    service_cidr        = "10.100.0.0/16"
    dns_service_ip      = "10.100.0.10"
  }

  key_vault_secrets_provider {
    secret_rotation_enabled = true
  }

  tags = var.tags
}

resource "azurerm_kubernetes_cluster_node_pool" "gpu" {
  count = var.gpu_node_pool_enabled ? 1 : 0

  name                  = "gpu"
  kubernetes_cluster_id = azurerm_kubernetes_cluster.this.id
  vm_size               = var.gpu_node_pool.vm_size
  vnet_subnet_id        = module.vnet.subnets["aks"].resource_id
  os_sku                = "Ubuntu"
  mode                  = "User"
  auto_scaling_enabled  = true
  min_count             = var.gpu_node_pool.min_count
  max_count             = var.gpu_node_pool.max_count
  node_count            = max(var.gpu_node_pool.min_count, 1)

  node_taints = ["nvidia.com/gpu=present:NoSchedule"]
  node_labels = { workload = "gpu" }

  tags = var.tags
}

## Exercise
Disable the GPU node pool for a CPU-only environment and verify your plan changes.

In [ ]:
# TODO: In terraform.tfvars set gpu_node_pool_enabled = false
# TODO: Run terraform plan and confirm no gpu node pool resource is created.
print('Tip: This keeps Ollama demos working while waiting for GPU quota.')

---
**Next:** [Lesson 2.2 - Key Vault And Kubernetes RBAC Bridge](./lesson-2.2-key-vault-and-kubernetes-rbac-bridge.ipynb)